# RTIC P&ID Ingestion - Catalyst Test Bed Facility

**Goal:** ingest the Catalyst Test Bed P&ID into machine-readable datasets that can later feed an RTIC engineering knowledge graph.

Pipeline:

`PDF/DWG -> page rendering -> text/tag extraction -> line/symbol candidates -> equipment/instrument/valve datasets -> graph nodes/edges -> validation -> Neo4j/NetworkX`

This notebook deliberately separates **ground-truth / seed labels** from **machine candidates**. It does not invent process connectivity when the drawing has not yet been validated.


In [ ]:
!pip -q install pymupdf opencv-python-headless pandas numpy networkx openpyxl
import os, re, math, zipfile, shutil
from pathlib import Path
import fitz, cv2, numpy as np, pandas as pd, networkx as nx
from google.colab import files
print("Libraries ready")

## 1. Upload the as-built P&ID PDF

Upload: `SPL_2024-25_PID_R05_As Built (Catalyst Test Bed Facility).pdf`

Later, add the native DWG/DXF and previous revisions for stronger ingestion and revision-diffing.


In [ ]:
uploaded = files.upload()
pdf_name = next(iter(uploaded))
PDF_PATH = f"/content/{pdf_name}"
OUT = Path("/content/RTIC_PID_Ingestion_Output")
(OUT/"datasets").mkdir(parents=True, exist_ok=True)
(OUT/"page_images").mkdir(parents=True, exist_ok=True)
print(PDF_PATH)

## 2. Extract page text, coordinates and render pages

In [ ]:
doc = fitz.open(PDF_PATH)
pages, spans = [], []
for pno, page in enumerate(doc, start=1):
    txt = page.get_text("text")
    pages.append({"page":pno,"width_pt":page.rect.width,"height_pt":page.rect.height,
                  "text_char_count":len(txt),"raw_text":txt})
    d=page.get_text("dict")
    for b in d.get("blocks",[]):
        if b.get("type") != 0: continue
        for ln in b.get("lines",[]):
            for sp in ln.get("spans",[]):
                t=sp.get("text","").strip()
                if t:
                    x0,y0,x1,y1=sp["bbox"]
                    spans.append({"page":pno,"text":t,"x0_pt":x0,"y0_pt":y0,
                                  "x1_pt":x1,"y1_pt":y1,"font":sp.get("font"),
                                  "font_size":sp.get("size")})
    pix=page.get_pixmap(matrix=fitz.Matrix(1.8,1.8), alpha=False)
    pix.save(str(OUT/"page_images"/f"page_{pno:02d}.png"))
pd.DataFrame(pages).to_csv(OUT/"datasets"/"document_pages.csv",index=False)
pd.DataFrame(spans).to_csv(OUT/"datasets"/"text_spans.csv",index=False)
print("Rendered pages:", len(doc))

## 3. Detect candidate line segments

These are **candidates only**. Later a classifier/rule layer separates process lines, pneumatic/electrical signals, borders, table lines and equipment strokes.


In [ ]:
line_rows=[]
for imgfile in sorted((OUT/"page_images").glob("page_*.png")):
    pno=int(imgfile.stem.split("_")[-1])
    im=cv2.imread(str(imgfile))
    gray=cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
    edges=cv2.Canny(gray,60,160)
    lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=90,minLineLength=45,maxLineGap=8)
    if lines is None: continue
    for i,l in enumerate(lines[:5000]):
        x1,y1,x2,y2=map(int,l[0])
        line_rows.append({"page":pno,"candidate_id":f"P{pno:02d}_L{i:04d}",
                          "x1":x1,"y1":y1,"x2":x2,"y2":y2,
                          "length_px":float(np.hypot(x2-x1,y2-y1)),
                          "angle_deg":float(np.degrees(np.arctan2(y2-y1,x2-x1)))})
pd.DataFrame(line_rows).to_csv(OUT/"datasets"/"line_candidates.csv",index=False)
print("line candidates:",len(line_rows))

## 4. Seed equipment + section ground truth

For the first PoC, use reviewed labels as supervised ground truth.  
Do **not** treat OCR/CV output as engineering truth until validated.


In [ ]:
equipment = pd.DataFrame([
[3,"V-200","Feed Tank","Liquid Feed Section"],
[3,"HE-200","Feed Pre-Heater","Liquid Feed Section"],
[3,"SM-200","Static Mixer","Liquid Feed Section"],
[3,"FRN-200","Tube Furnace","Liquid Feed Section"],
[4,"R-300","Single Tube Reactor","Reactor-1 Section"],
[4,"FRN-300","Four Zone Furnace","Reactor-1 Section"],
[5,"R-400","Jacketed Tube Reactor","Reactor-2 Section"],
[5,"CH-400","Chiller","Reactor-2 Section"],
[5,"HE-400","Product Cooler","Reactor-2 Section"],
[5,"HE-401","Heat Exchanger","Reactor-2 Section"],
[5,"HE-402","Heat Exchanger","Reactor-2 Section"],
[5,"THY-400","Thermosiphon Arrangement","Reactor-2 Section"],
[6,"V-500","Gas Liquid Separator","Product Section"],
[6,"V-501","Product Storage Tank","Product Section"],
[7,"HE-600","Sample Cooler","Sample Handling System 1"],
[7,"V-600","Knockout Pot","Sample Handling System 1"],
[8,"HE-700","Sample Cooler","Sample Handling System 1"],
], columns=["page","tag","equipment_type","section"])
equipment["human_review_required"]=True
equipment.to_csv(OUT/"datasets"/"equipment_seed.csv",index=False)
equipment

## 5. Candidate tag extraction

Vector PDFs often store some labels as paths, so regex extraction alone will miss many tags.  
Use this as a first pass; next phase adds symbol detection and OCR only for unresolved regions.


In [ ]:
spans=pd.read_csv(OUT/"datasets"/"text_spans.csv")
pat=re.compile(r"\b[A-Z]{1,6}(?:-[A-Z]{0,4})?-\d{2,4}\b")
rows=[]
for _,r in spans.iterrows():
    for tag in pat.findall(str(r.text).replace(" ","")):
        rows.append({"page":r.page,"tag":tag,"source":"vector_text_regex"})
for _,r in equipment.iterrows():
    rows.append({"page":r.page,"tag":r.tag,"source":"reviewed_seed"})
tags=pd.DataFrame(rows).drop_duplicates()
tags.to_csv(OUT/"datasets"/"tag_candidates.csv",index=False)
tags.head(30)

## 6. Build first knowledge-graph seed

At this stage we add only relationships we can defend, such as **section CONTAINS equipment**.  
Process `CONNECTED_TO` edges should be generated only after line-to-symbol topology is validated.


In [ ]:
G=nx.MultiDiGraph()
for _,r in equipment.iterrows():
    sec=r.section
    G.add_node(sec,node_type="Section")
    G.add_node(r.tag,node_type="Equipment",equipment_type=r.equipment_type,page=int(r.page))
    G.add_edge(sec,r.tag,relationship="CONTAINS")
nx.write_graphml(G, OUT/"datasets"/"knowledge_graph_seed.graphml")
print("nodes",G.number_of_nodes(),"edges",G.number_of_edges())

## 7. Next ingestion stages

1. Train/correct symbol detector using page-10 legend.
2. Extract valves/instruments with bounding boxes.
3. Skeletonize process and signal lines.
4. Snap line endpoints to symbols/nozzles.
5. Resolve off-page connectors.
6. Create `CONNECTED_TO`, `MEASURES`, `CONTROLS`, `SIGNAL_TO` relationships.
7. Join valve list / instrument index / I/O / cable schedule / BOM.
8. Validate against EIL/Thermax rules.
9. Load graph into Neo4j.
10. Compare Rev-00...Rev-04 and create change-impact datasets.


## 8. Save/download all generated datasets

In [ ]:
zip_path="/content/RTIC_PID_Ingestion_Output.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for root,_,fnames in os.walk(OUT):
        for f in fnames:
            p=os.path.join(root,f)
            z.write(p,arcname=os.path.relpath(p,OUT))
files.download(zip_path)
print(zip_path)